## Generate random graphs
1. Sizes n = 100, 125, ..., 500 (17 values)
2. 20 graphs for 100 =< n =< 200, 15 graphs for 225≤n≤350, 10 graphs for 375≤n≤500 = 250 graphs
3. Random trees x 1, random graphs p = 0.3 (main), 0.1, 0.5 (robustness)

Results in folder: test_graphs_all/datasets, file names "trees", "rand_01", "rand_03", "rand_05"

In [1]:
import networkx as nx
import pandas as pd
import random
from cup_gather_data import hcgcr_data
import json
from tqdm import tqdm
from pathlib import Path

base_path = Path(r'C:\Users\korol\OneDrive\Pulpit\Master\CUP - remove asymmetric, remove orbits, counting changes\test_graphs\dataset_31_01')

In [2]:
# Function
def coloring_dict(G):
    iterations = hcgcr_data(G)
    data = {
        "n": G.number_of_nodes(),
        "graph": json.dumps(list(G.edges())),
        "nr_iter": len(iterations),
        "coloring": json.dumps([
            {"color": it.coloring["color"].to_list(),
                "hash": it.coloring["hash"].to_list()}
            for it in iterations
        ]),
        "hash_dict": json.dumps([it.hash_dict for it in iterations], default=list)
    }
    return data

n_values = list(range(100, 501, 25))

def num_graphs_for_n(n):
    if n <= 200:
        return 20
    elif n <= 350:
        return 15
    else:
        return 10

In [7]:
# Trees
print("Generating random trees")
n_values = list(range(100, 501, 25))
result = []
for n in n_values:
    size = num_graphs_for_n(n)
    for _ in tqdm(range(size), desc=f"Generating trees (n={n})"):
        G = nx.random_unlabeled_tree(n)
        result.append(coloring_dict(G))
    
df = pd.DataFrame(result)
df.to_csv(base_path / f"trees.csv", index=False)

Generating random trees


Generating trees (n=500): 100%|██████████| 10/10 [04:03<00:00, 24.39s/it]


In [9]:
# Random graphs
def generate_random_graphs(p):
    print(f"Generating random graphs, p={p}")
    n_values = list(range(100, 501, 25))
    result = []
    for n in n_values:
        size = num_graphs_for_n(n)
        for _ in tqdm(range(size), desc=f"Generating graphs (n={n})"):
            G = nx.fast_gnp_random_graph(n, p)
            result.append(coloring_dict(G))
    df = pd.DataFrame(result)
    df.to_csv(base_path / f"rand_0{str(10*p)}.csv", index=False)

In [10]:
# Random graphs for p = 0.1
generate_random_graphs(0.1)

Generating random graphs, p=0.1


Generating graphs (n=500): 100%|██████████| 10/10 [01:36<00:00,  9.64s/it]


In [11]:
# Random graphs for p = 0.3
generate_random_graphs(0.3)

Generating random graphs, p=0.3


Generating graphs (n=500): 100%|██████████| 10/10 [01:17<00:00,  7.77s/it]


In [12]:
# Random graphs for p = 0.5
generate_random_graphs(0.5)

Generating random graphs, p=0.5


Generating graphs (n=500): 100%|██████████| 10/10 [01:32<00:00,  9.27s/it]


### Generate random graphs 31.01
1. trees: balanced binary trees h = 5,6,7,8
2. trees: labeled trees (100-500) x250
3. RG: fast ... (100-500) p = 0.01, 0.03, 0.05
4. epinions 

In [4]:
# 1 balanced trees
print("Generating balanced trees")
result = []
for h in tqdm([5,6,7,8,9], desc="Generating trees"):
    G = nx.balanced_tree(r=2, h=h)
    result.append(coloring_dict(G))
    
df = pd.DataFrame(result)
df.to_csv(base_path / f"balanced_trees.csv", index=False)

Generating balanced trees


Generating trees: 100%|██████████| 5/5 [00:00<00:00, 50.23it/s]


In [6]:
# 2 labaled trees
seed = "0000"
print("Generating random trees")
result = []
for _ in tqdm(range(250), desc="Generating trees"):
    n = random.randint(100, 501)
    G = nx.random_labeled_tree(n)
    result.append(coloring_dict(G))
    
df = pd.DataFrame(result)
df = df.sort_values(by="n")
df.to_csv(base_path / f"random_trees.csv", index=False)

Generating random trees


Generating trees: 100%|██████████| 250/250 [01:19<00:00,  3.16it/s]


In [7]:
# 3 random graphs
print("Generating random graphs")
for p,st in zip([0.005, 0.01, 0.03, 0.05], ["05","1","3","5"]):
    print("p = ", p)
    result = []
    for _ in tqdm(range(250), desc="Generating graphs"):
        n = random.randint(100, 501)
        G = nx.fast_gnp_random_graph(n, p)
        result.append(coloring_dict(G))
    df = pd.DataFrame(result)
    df = df.sort_values(by="n")
    df.to_csv(base_path / f"rg_p_{st}.csv", index=False)

Generating random graphs
p =  0.005


Generating graphs: 100%|██████████| 250/250 [00:46<00:00,  5.41it/s]


p =  0.01


Generating graphs: 100%|██████████| 250/250 [00:42<00:00,  5.83it/s]


p =  0.03


Generating graphs: 100%|██████████| 250/250 [00:34<00:00,  7.35it/s]


p =  0.05


Generating graphs: 100%|██████████| 250/250 [00:33<00:00,  7.54it/s]


In [4]:
# 4 Epinions
G = nx.read_edgelist(
    Path(base_path / "soc-Epinions1.txt"),
    create_using=nx.Graph(),
    delimiter="\t",
    comments="#",
    nodetype=int
)
G = nx.convert_node_labels_to_integers(G, ordering="sorted")
result = coloring_dict(G)
df = pd.DataFrame([result])
df.to_csv(base_path / f"epinions.csv", index=False)

### Calculate results
![alt text](image.png)

In [ ]:
import math
import random

def generate_G_new(G, beta=0.05, alpha=0.10, seed=None):
    """
    1) Choose S of size ceil(beta*n)
    2) Toggle k edges among pairs within S, where:
        k = ceil(alpha * C(|S|,2))
        AND ensure every vertex in S appears in at least one toggled pair.
    Args:
        G: original graph
        beta: fraction of n_G that will be in S
        alpha: fraction of edges in S that will be changed

    Returns (G_new, S, changed_pairs)
    """
    rng = random.Random(seed)
    G_new = G.copy()
    nodes = list(G_new.nodes())
    n = len(nodes)

    if n < 2:
        return G_new, set(nodes), set()

    # --- choose S ---
    s_size = max(2, math.ceil(beta * n))   # at least 2 so there are internal pairs
    s_size = min(s_size, n)
    S = set(rng.sample(nodes, s_size))
    S_list = list(S)

    # --- total possible internal pairs ---
    M = s_size * (s_size - 1) // 2

    # --- target number of changes ---
    k = max(1, math.ceil(alpha * M))

    # Minimum to cover all vertices: ceil(s_size/2)
    k_min_cover = (s_size + 1) // 2
    k = max(k, k_min_cover)
    k = min(k, M)

    changed_pairs = set()

    # --- Step 1: enforce coverage (pair up vertices) ---
    S_mixed = S_list[:]
    rng.shuffle(S_mixed)

    for i in range(0, s_size - 1, 2):
        u, v = S_mixed[i], S_mixed[i + 1]
        a, b = (u, v) if u < v else (v, u)
        changed_pairs.add((a, b))

    # If s is odd, last vertex hasn't been included yet; connect it to a random other vertex
    if s_size % 2 == 1:
        last = S_mixed[-1]
        other = rng.choice(S_mixed[:-1])
        a, b = (last, other) if last < other else (other, last)
        changed_pairs.add((a, b))

    # --- Step 2: add additional random internal pairs until reach k ---
    # Precompute all internal pairs for sampling
    all_pairs = []
    for i in range(s_size):
        for j in range(i + 1, s_size):
            u, v = S_list[i], S_list[j]
            all_pairs.append((u, v) if u < v else (v, u))

    # Fill up to k with distinct pairs
    # (simple loop is fine at your sizes)
    # if a pair was in changed_pairs the len will not grow
    while len(changed_pairs) < k:
        a, b = rng.choice(all_pairs)
        changed_pairs.add((a, b))

    # --- Apply toggles ---
    for u, v in changed_pairs:
        if G_new.has_edge(u, v):
            G_new.remove_edge(u, v)
        else:
            G_new.add_edge(u, v)

    return G_new, S, changed_pairs

################ GENERATE UPDATED GRAPHS #####################

def random_graphs_up(s_p):
    out = {}
    for name, df in test_graphs.items():
        ups = []
        for row in tqdm(df.itertuples(index=False), desc="Generating for the file: "):
            G = row.graph
            s_len = max(1, math.ceil(G.number_of_edges() * s_p))
            G_up, S = change_random_set_of_edges(G, s_len)
            ups.append({"graph_up": G_up, "S": S})
        out[name] = ups
    return out

def update_dict(G, S, iterations_up, q_lens):
    avg = (sum(q_lens) / len(q_lens)) if q_lens else 0.0
    data = {
        "n": G.number_of_nodes(),
        "m": G.number_of_edges(),
        "graph": json.dumps(list(G.edges())),
        "S": json.dumps(list(S)),
        "q_lens": json.dumps(q_lens),
        "q_len_average": avg,
        "coloring": json.dumps([
            {"color": it.coloring["color"].to_list(),
                "hash": it.coloring["hash"].to_list()}
            for it in iterations_up
        ]),
        "hash_dict": json.dumps([it.hash_dict for it in iterations_up], default=list)
    }
    return data